# Pattern Discrimination and Classification of Neural Population Data

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* [`matlab/ClassificationTutorial.m`](../matlab/ClassificationTutorial.m).

This tutorial asks a question PCA cannot answer. PCA finds the directions along which a
population varies the most — but variance is not the same thing as usefulness. If you want
to know *which of two stimuli was presented on a given trial*, you need directions that
**separate** the two response distributions, and the most separating direction is often not
the highest-variance one.

We build up from a single neuron to a full population, and from a simple threshold rule to
regularized, cross-validated multivariate classifiers.

| Part | Topic |
|---|---|
| I | Signal detection theory: one neuron, ROC, AUC, and $d'$ |
| II | Neurometric functions: which neurons carry the information? |
| III | Linear discriminant analysis: from one neuron to many |
| IV | Logistic regression (a GLM classifier) |
| V | Support vector machines and the margin |
| VI | Cross-validation, overfitting, and regularization |
| VII | Decoding geometry: signal vs. noise correlations |
| VIII | Nonlinear classifiers and a head-to-head comparison |

Throughout we use the same simulated visual population as the population PCA tutorial —
neurons with von Mises orientation tuning — now with realistic trial-to-trial variability.
The task is **fine orientation discrimination**: on each trial the animal sees a grating at
either $90 - \Delta\theta/2$ or $90 + \Delta\theta/2$ degrees and must report which.

**A note on style.** Every core method here — ROC, $d'$, LDA, IRLS, cross-validation — is
implemented from scratch first, because the mechanism *is* the lesson. We then check each
against the scikit-learn equivalent, so you can see both that the implementation is right
and which library call to reach for in your own work.

**Prerequisites:** the linear algebra tutorial, `PCANeuroPopTutorial`, and the stochastic
processes tutorial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg
from scipy.stats import rankdata, norm

rng = np.random.default_rng(7)     # fixed seed, so every run reproduces the figures

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 11, "font.size": 9,
})

BLUE, RED, GREEN, PURPLE, GREY = "#3366d9", "#d94d3f", "#33914f", "#6a4fa8", "#808080"

# np.trapz was renamed np.trapezoid in NumPy 2.0; accept either.
trapezoid = getattr(np, "trapezoid", None) or np.trapz

---
## Part 0. The simulated population and the discrimination task

The encoding model is deliberately simple, because the subject of this tutorial is the
**decoder**, not the encoder. Each neuron has a von Mises tuning curve, periodic with 180°:

$$\mu_i(\theta) = \text{baseline} + \text{gain}\cdot
  \frac{\exp[\kappa\cos(2(\theta - \phi_i))]}{\exp(\kappa)}$$

Dividing by $\exp(\kappa)$ normalizes the peak to 1, so `gain` is interpretable as the peak
evoked rate.

What is new here — and what makes classification meaningful at all — is **noise**. Real
neurons do not produce the same response twice. We use the standard cortical approximation
that variance grows in proportion to the mean,

$$\sigma_i^2 = F \cdot \mu_i$$

where $F$ is the Fano factor. $F = 1$ is Poisson; values slightly above 1 are typical in
visual cortex.

In [ ]:
N        = 80      # number of neurons
kappa    = 3.0     # orientation tuning concentration
baseline = 2.0     # baseline rate (spikes/s)
gain     = 10.0    # peak evoked rate above baseline
fano     = 1.5     # variance-to-mean ratio

pref_oris = np.linspace(0, 180, N, endpoint=False)   # preferred orientations

def tuning_mean(pref_oris, kappa, stim_ori, baseline=baseline, gain=gain):
    '''Von Mises orientation tuning, 180-deg periodic, peak normalized to 1.'''
    d = np.deg2rad(stim_ori) - np.deg2rad(np.asarray(pref_oris, float))
    return baseline + gain * np.exp(kappa * np.cos(2 * d)) / np.exp(kappa)

def sample_trials(mu, n_trials, fano, rng, chol_lower=None):
    '''Draw n_trials samples with mean mu and variance fano * mu.

    chol_lower is the LOWER Cholesky factor of the desired noise correlation matrix
    (pass None for independent noise). Watch the convention: np.linalg.cholesky
    returns the LOWER factor L with L @ L.T = C, whereas MATLAB's chol returns the
    UPPER factor R with R' * R = C. So here we post-multiply by L.T, not by L.
    '''
    mu = np.atleast_1d(np.asarray(mu, float))
    z = rng.standard_normal((n_trials, mu.size))
    if chol_lower is not None:
        z = z @ chol_lower.T                       # cov(z) = L @ L.T = C
    sd = np.sqrt(np.maximum(fano * mu, 1e-12))
    return np.maximum(mu + z * sd, 0.0)            # rates cannot be negative

### The discrimination task

The separation below was chosen so the problem sits in the interesting regime: hard enough
that no single neuron solves it, easy enough that the population does not saturate at 100%.
If you make the task too easy, every method below scores 1.0 and the comparisons become
vacuous — a failure mode worth remembering when you design your own simulations.

In [ ]:
theta_center = 90.0
dtheta       = 4.0
ori_A, ori_B = theta_center - dtheta/2, theta_center + dtheta/2   # 88 and 92 deg
n_trials     = 400

mu_A = tuning_mean(pref_oris, kappa, ori_A)
mu_B = tuning_mean(pref_oris, kappa, ori_B)

# Parts I-VI use INDEPENDENT noise across neurons. Part VII relaxes that,
# and you will see that the assumption matters enormously.
X_A = sample_trials(mu_A, n_trials, fano, rng)
X_B = sample_trials(mu_B, n_trials, fano, rng)

# Standard supervised-learning layout: rows are trials, columns are neurons.
X = np.vstack([X_A, X_B])
y = np.concatenate([np.zeros(n_trials, int), np.ones(n_trials, int)])

print(f"Design matrix X is {X.shape[0]} trials x {X.shape[1]} neurons")
print(f"Task: {ori_A:g} deg vs {ori_B:g} deg (separation {dtheta:g} deg)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

axes[0].plot(pref_oris, mu_A, lw=2, color=BLUE, label=f"{ori_A:g}°")
axes[0].plot(pref_oris, mu_B, lw=2, color=RED,  label=f"{ori_B:g}°")
axes[0].set(ylabel="Mean rate (spikes/s)", title="Mean population response to the two stimuli")
axes[0].legend()

axes[1].plot(pref_oris, mu_B - mu_A, "k", lw=2)
axes[1].axhline(0, color=GREY, ls=":")
axes[1].set(xlabel="Preferred orientation (deg)", ylabel="Δ mean rate", xlim=(0, 180),
            title="Difference between the two mean patterns — the SIGNAL")
axes[1].set_xticks(np.arange(0, 181, 30))
fig.tight_layout()

Study the lower panel. The two mean patterns are nearly identical — a 4° shift of a broad
tuning curve barely moves anything. And the difference is largest **not** at 90°, where both
stimuli drive the neuron equally hard, but on the **flanks**, where the tuning curve is
steepest.

This is the single most important intuition in neural decoding: *information lives in the
slope, not the peak*. The neuron that responds most is not the neuron that informs most.

> ### Homework question 1
> **(a)** How does the difference curve change as $\Delta\theta$ grows from 2° to 45°? At
> what separation does it stop being well approximated by the derivative of the tuning curve?
>
> **(b)** The difference curve has zero crossings. Where are they, and what does a neuron
> sitting exactly at one contribute to the discrimination?
>
> **(c)** If you doubled $\kappa$ (sharper tuning), would the peak of the difference curve
> get larger or smaller? Would it move?

---
## Part I. One neuron: signal detection theory, ROC, and $d'$

Before touching the population, solve the problem for a single neuron — because every
multivariate method in this tutorial reduces to exactly this problem after projecting onto
one axis.

In [ ]:
# The neuron whose mean response is most ENHANCED by stimulus B. (The neuron most
# enhanced by A is equally informative; we take the B-preferring one so the ROC curve
# bows above the diagonal, which is how it is conventionally drawn.)
best_neuron = int(np.argmax(mu_B - mu_A))
print(f"Most informative single neuron: #{best_neuron} "
      f"(preferred ori = {pref_oris[best_neuron]:.1f} deg)")

rA = X_A[:, best_neuron]
rB = X_B[:, best_neuron]

An ideal observer watching only this neuron must choose a **criterion** $c$ and report "B"
whenever the response exceeds it. Every choice of $c$ trades off two quantities:

- **hit rate** — $P(\text{report B} \mid \text{stimulus was B})$
- **false alarm rate** — $P(\text{report B} \mid \text{stimulus was A})$

A low criterion catches every B trial but mislabels many A trials; a high one is
conservative in both directions. Neither is *the* answer — the full trade-off curve **is**
the answer, and that curve is the Receiver Operating Characteristic.

In [ ]:
def roc_curve_manual(r0, r1):
    '''ROC by explicit criterion sweep. r0 is the class-0 (noise) distribution.'''
    crit = np.unique(np.concatenate([r0, r1]))[::-1]
    crit = np.concatenate([[crit[0] + 1], crit])       # start above every observation
    phit = np.array([(r1 >= c).mean() for c in crit])
    pfa  = np.array([(r0 >= c).mean() for c in crit])
    return pfa, phit, trapezoid(phit, pfa)

def auc_from_ranks(r0, r1):
    '''AUC = P(r1 > r0) + 0.5 P(r1 == r0), via the Mann-Whitney rank statistic.'''
    n0, n1 = len(r0), len(r1)
    ranks = rankdata(np.concatenate([r0, r1]))         # average ranks for ties
    return (ranks[n0:].sum() - n1 * (n1 + 1) / 2) / (n0 * n1)

pfa, phit, auc = roc_curve_manual(rA, rB)
dprime = (rB.mean() - rA.mean()) / np.sqrt((rA.var(ddof=1) + rB.var(ddof=1)) / 2)

print(f"AUC by trapezoidal integration : {auc:.4f}")
print(f"AUC by rank statistic          : {auc_from_ranks(rA, rB):.4f}")
print(f"d-prime                        : {dprime:.4f}")
print(f"AUC predicted from d-prime     : {norm.cdf(dprime / np.sqrt(2)):.4f}")

The area under the ROC curve summarizes the whole trade-off in one number, and it has an
interpretation with nothing to do with criteria: **AUC is exactly the probability that a
randomly chosen B-trial response exceeds a randomly chosen A-trial response.** That is
precisely the performance of an ideal observer in a two-alternative forced choice task. The
two computations above agree to four decimals, as they must.

If both distributions were Gaussian with equal variance, the entire ROC would be fixed by a
single number, the separation of the means in units of the common standard deviation:

$$d' = \frac{\mu_B - \mu_A}{\sqrt{(\sigma_A^2 + \sigma_B^2)/2}},
\qquad \text{AUC} = \Phi\!\left(\frac{d'}{\sqrt 2}\right)$$

The agreement above is good but not exact, because our responses are *not* equal-variance
Gaussians — variance scales with the mean, and rates are clipped at zero. A general lesson:
AUC is nonparametric and always valid, while $d'$ buys interpretability at the cost of a
distributional assumption.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 7))

# --- overlapping distributions, with three example criteria ---
edges = np.linspace(min(rA.min(), rB.min()), max(rA.max(), rB.max()), 40)
axes[0,0].hist(rA, edges, color=BLUE, alpha=0.5, label=f"{ori_A:g}°")
axes[0,0].hist(rB, edges, color=RED,  alpha=0.5, label=f"{ori_B:g}°")
crit_examples = np.quantile(np.concatenate([rA, rB]), [0.25, 0.5, 0.75])
for c in crit_examples:
    axes[0,0].axvline(c, color="k", ls="--", lw=1)
axes[0,0].set(xlabel="Response (spikes/s)", ylabel="Number of trials",
              title=f"Neuron {best_neuron}: overlapping distributions")
axes[0,0].legend()

# --- the ROC curve ---
axes[0,1].plot(pfa, phit, "k", lw=2)
axes[0,1].plot([0,1], [0,1], ":", color=GREY)
for c in crit_examples:
    axes[0,1].plot((rA >= c).mean(), (rB >= c).mean(), "o", ms=8,
                   mfc="#f2c14e", mec="k")
axes[0,1].set(xlabel="False alarm rate  P(report B | A)", ylabel="Hit rate  P(report B | B)",
              xlim=(0,1), ylim=(0,1), aspect="equal", title=f"ROC curve, AUC = {auc:.3f}")

# --- the AUC / d' relationship ---
d_grid = np.linspace(0, 4, 200)
axes[1,0].plot(d_grid, norm.cdf(d_grid/np.sqrt(2)), "k", lw=2)
axes[1,0].plot(dprime, auc, "o", ms=9, mfc=RED, mec="k")
axes[1,0].set(xlabel="d'", ylabel="AUC", ylim=(0.5, 1),
              title="AUC = Φ(d'/√2) for equal-variance Gaussians")

# --- accuracy depends on criterion; AUC does not ---
crit_grid = np.linspace(rA.min(), rB.max(), 200)
acc_grid = [((rB >= c).mean() + (rA < c).mean()) / 2 for c in crit_grid]
axes[1,1].plot(crit_grid, acc_grid, "k", lw=2, label="accuracy vs. criterion")
axes[1,1].axhline(auc, ls="--", color=RED, label="AUC")
axes[1,1].set(xlabel="Criterion (spikes/s)", ylabel="Balanced accuracy", ylim=(0.4, 1),
              title="Accuracy depends on criterion; AUC does not")
axes[1,1].legend(loc="lower center", fontsize=8)
fig.tight_layout()

The best accuracy achievable by *any* criterion is close to, but generally not equal to, the
AUC; they coincide only in the equal-variance Gaussian case. Neurophysiologists report AUC
rather than percent correct precisely because it removes the experimenter's arbitrary choice
of criterion.

### Checking against scikit-learn

Our hand-written ROC should agree with the library implementation exactly. It is worth
running this kind of check whenever you implement something yourself.

In [ ]:
from sklearn.metrics import roc_curve as sk_roc_curve, roc_auc_score

labels = np.concatenate([np.zeros(len(rA), int), np.ones(len(rB), int)])
scores = np.concatenate([rA, rB])
fpr_sk, tpr_sk, _ = sk_roc_curve(labels, scores)
auc_sk = roc_auc_score(labels, scores)

print(f"AUC, ours       : {auc:.6f}")
print(f"AUC, sklearn    : {auc_sk:.6f}")
print(f"max |difference|: {abs(auc - auc_sk):.2e}")

fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.plot(pfa, phit, "k", lw=3, alpha=0.4, label="ours (criterion sweep)")
ax.plot(fpr_sk, tpr_sk, "--", color=RED, lw=1.5, label="sklearn.metrics.roc_curve")
ax.plot([0,1],[0,1], ":", color=GREY)
ax.set(xlabel="False alarm rate", ylabel="Hit rate", aspect="equal", title="Same curve")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()

> ### Homework question 2
> **(a)** Show algebraically that $\text{AUC} = P(r_B > r_A)$ by interpreting the ROC
> integral as an expectation over criteria.
>
> **(b)** What is the AUC if the two distributions are identical? What if the neuron fires
> *less* for B than for A? Why do neurophysiologists sometimes report
> $\max(\text{AUC}, 1-\text{AUC})$?
>
> **(c)** Set `fano = 0` at the top and re-run. What happens to the ROC, and why is the
> resulting "perfect" performance scientifically uninteresting?
>
> **(d)** The ROC curve is jagged. What sets the size of the steps, and how would you get a
> smooth curve?

---
## Part II. Neurometric functions: which neurons carry the information?

We chose the best neuron by looking at the difference of means. Now compute the AUC for
**every** neuron and see how discriminability is distributed across the population.

In [ ]:
auc_all = np.array([auc_from_ranks(X_A[:, i], X_B[:, i]) for i in range(N)])

# Analytic tuning-curve slope at the reference orientation, for comparison
h = 0.01
tc_slope = (tuning_mean(pref_oris, kappa, theta_center + h, 0, gain)
            - tuning_mean(pref_oris, kappa, theta_center - h, 0, gain)) / (2*h)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(pref_oris, auc_all, "k", lw=1.5)
axes[0].axhline(0.5, color=GREY, ls=":")
axes[0].plot(pref_oris[best_neuron], auc_all[best_neuron], "o", ms=8, mfc="#f2c14e", mec="k")
axes[0].set(xlabel="Preferred orientation (deg)", ylabel="AUC", xlim=(0, 180),
            title="Single-neuron discriminability across the population")

axes[1].plot(pref_oris, np.abs(auc_all - 0.5)/np.abs(auc_all - 0.5).max(), "k", lw=2,
             label="|AUC − 0.5|")
axes[1].plot(pref_oris, np.abs(tc_slope)/np.abs(tc_slope).max(), "--", lw=2, color=RED,
             label="|tuning-curve slope|")
axes[1].set(xlabel="Preferred orientation (deg)", ylabel="Normalized magnitude", xlim=(0, 180),
            title="Discriminability tracks tuning-curve SLOPE")
axes[1].legend(fontsize=8)
for a in axes: a.set_xticks(np.arange(0, 181, 30))
fig.tight_layout()

Two things deserve attention. Neurons tuned to exactly 90° — the ones that respond *most
strongly* to both stimuli — are the **least** informative, with AUC near 0.5, because both
stimuli sit symmetrically on their tuning peak. And AUC dips below 0.5 for half the
population, which simply means those neurons fire more for A than for B; information is
information regardless of sign, and a downstream decoder can flip the weight.

The overlay makes it quantitative: single-neuron discriminability for a fine discrimination
is proportional to the *derivative* of the tuning curve at the reference orientation,
divided by the noise standard deviation.

### The neurometric function

Now sweep the difficulty. For each angular separation, ask how well the best single neuron
performs and how well the whole population performs. These curves are **neurometric
functions**, directly comparable to a psychometric function measured behaviorally.

In [ ]:
dtheta_grid = np.array([1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 45], float)
auc_best = np.zeros(dtheta_grid.size)
auc_pop  = np.zeros(dtheta_grid.size)
n_tr_neuro = 300

for k, dth in enumerate(dtheta_grid):
    mA = tuning_mean(pref_oris, kappa, theta_center - dth/2)
    mB = tuning_mean(pref_oris, kappa, theta_center + dth/2)
    XA = sample_trials(mA, n_tr_neuro, fano, rng)
    XB = sample_trials(mB, n_tr_neuro, fano, rng)

    a = np.array([auc_from_ranks(XA[:, i], XB[:, i]) for i in range(N)])
    auc_best[k] = max(a.max(), 1 - a.min())

    w = mB - mA                                    # difference-of-means readout
    auc_pop[k] = auc_from_ranks(XA @ w, XB @ w)

def threshold_at(x, auc_vals, target=0.75):
    '''Linearly interpolate the x at which AUC first reaches target.'''
    idx = np.argmax(auc_vals >= target)
    if not (auc_vals >= target).any(): return np.nan
    if idx == 0: return x[0]
    x0, x1, a0, a1 = x[idx-1], x[idx], auc_vals[idx-1], auc_vals[idx]
    return x0 + (target - a0) * (x1 - x0) / (a1 - a0)

print(f"75% threshold, best single neuron : {threshold_at(dtheta_grid, auc_best):.2f} deg")
print(f"75% threshold, population         : {threshold_at(dtheta_grid, auc_pop):.2f} deg")

In [ ]:
# How does pooling scale with the number of neurons?
n_sub = np.unique(np.round(np.logspace(0, np.log10(N), 12)).astype(int))
w_full = mu_B - mu_A
auc_sub = np.array([auc_from_ranks(X_A[:, sel] @ w_full[sel], X_B[:, sel] @ w_full[sel])
                    for sel in (np.round(np.linspace(0, N-1, n)).astype(int) for n in n_sub)])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogx(dtheta_grid, auc_best, "-o", lw=2, color=BLUE, label="best single neuron")
axes[0].semilogx(dtheta_grid, auc_pop,  "-s", lw=2, color=RED,  label="population readout")
axes[0].axhline(0.5, color=GREY, ls=":")
axes[0].set(xlabel="Orientation separation (deg)", ylabel="AUC", ylim=(0.45, 1.02),
            title="Neurometric functions")
axes[0].legend(loc="lower right", fontsize=8)

axes[1].semilogx(n_sub, auc_sub, "-o", lw=2, color=GREEN)
axes[1].axhline(0.5, color=GREY, ls=":")
axes[1].set(xlabel="Number of neurons pooled", ylabel="AUC", ylim=(0.45, 1.02),
            title="Pooling improves discriminability (independent noise)")
fig.tight_layout()

The population curve sits far to the left of the single-neuron curve: the population resolves
separations no individual neuron can. The gap between them is the value of pooling.

> ### Homework question 3
> **(a)** With independent noise, $d'$ for the pooled readout grows as $\sqrt{n}$. Derive
> this. Does the right-hand curve match? *(Hint: convert AUC back to $d'$ and re-plot.)*
>
> **(b)** A trained animal's behavioral threshold is typically a few times better than the
> best single neuron but far worse than this population readout. What does that tell you
> about the assumption of independent noise? Part VII returns to this.
>
> **(c)** Restrict the population readout to neurons preferring 60–70°, then to neurons
> preferring 88–92°. Explain the difference in performance.

---
## Part III. Linear discriminant analysis

Every linear classifier has the same form: pick a weight vector $\mathbf{w}$, project each
trial's population response onto it, and compare to a threshold — decide B if
$\mathbf{w}^\top\mathbf{r} + b > 0$. The methods differ only in how they choose
$\mathbf{w}$.

In Part II we used the simplest choice, $\mathbf{w} = \mu_B - \mu_A$, which points from one
mean to the other. That is optimal when the noise is isotropic. Fisher's linear discriminant
does better by maximizing the separation of the projected means *relative to* the projected
within-class variance:

$$\mathbf{w} \propto S^{-1}(\mu_B - \mu_A)$$

where $S$ is the pooled within-class covariance. The $S^{-1}$ does something intuitive: it
discounts directions in which the noise is large and amplifies those in which it is small.

### First, two dimensions

A word about what follows. If we plot two neurons' responses to the 88° and 92° stimuli, the
clouds sit almost exactly on top of each other. That is not a defect of the plot — it is the
point of the fine discrimination task. The best single neuron reaches AUC ≈ 0.6, so no
*pair* can separate these stimuli either, and any boundary would run through the middle of
one indistinguishable blob.

So for the two-dimensional **pictures** — here and in the SVM section — we switch to a
coarser 84°-vs-96° discrimination, chosen so the best pair classifies at about 80%: clearly
above chance, so the boundary is meaningful, but far from perfect, so the picture stays
honest about what real data looks like. Every population result stays on the fine task.

In [ ]:
ori_C, ori_D = theta_center - 6, theta_center + 6      # 84 and 96 deg
mu_C = tuning_mean(pref_oris, kappa, ori_C)
mu_D = tuning_mean(pref_oris, kappa, ori_D)

# These panels also carry a shared GAIN fluctuation: on some trials the whole
# population is more responsive (attention, arousal, state). This is the dominant
# mode of shared variability in cortex, and it is what makes the clouds tilted
# rather than round. Part VII studies its consequences properly.
gain_sd = 0.25
X_C = np.maximum(sample_trials(mu_C, n_trials, fano, rng)
                 * (1 + gain_sd*rng.standard_normal((n_trials, 1))), 0)
X_D = np.maximum(sample_trials(mu_D, n_trials, fano, rng)
                 * (1 + gain_sd*rng.standard_normal((n_trials, 1))), 0)

n1 = int(np.argmax(mu_D - mu_C))     # strongly prefers 96 deg
n2 = int(np.argmin(mu_D - mu_C))     # strongly prefers 84 deg
pair = [n2, n1]

P_A, P_B = X_C[:, pair], X_D[:, pair]

In [ ]:
def pooled_cov(X0, X1):
    n0, n1_ = len(X0), len(X1)
    return ((n0-1)*np.cov(X0, rowvar=False) + (n1_-1)*np.cov(X1, rowvar=False)) / (n0+n1_-2)

def lda_train(X0, X1, lam=0.0):
    '''Fisher linear discriminant with shrinkage toward a scaled identity.

    lam = 0 is plain LDA; lam = 1 reduces to the difference-of-means decoder.
    When there are fewer trials than neurons, S is singular; Cholesky then fails
    and we fall back to the pseudoinverse's minimum-norm solution. That solution
    still overfits badly -- which is the point of the learning curve in Part VI --
    but it does so without numerical garbage.
    '''
    mu0, mu1 = X0.mean(axis=0), X1.mean(axis=0)
    S = pooled_cov(X0, X1)
    if lam > 0:
        S = (1 - lam)*S + lam*np.mean(np.diag(S))*np.eye(S.shape[0])
    try:
        w = scipy.linalg.cho_solve(scipy.linalg.cho_factor(S), mu1 - mu0)
    except (np.linalg.LinAlgError, scipy.linalg.LinAlgError):
        w = np.linalg.pinv(S) @ (mu1 - mu0)
    return w, -w @ (mu0 + mu1) / 2

def lda_predict(Xtr, ytr, Xte, lam=0.0):
    w, b = lda_train(Xtr[ytr == 0], Xtr[ytr == 1], lam)
    return (Xte @ w + b) > 0

def make_folds(n, k, rng):
    fold = np.empty(n, int)
    fold[rng.permutation(n)] = np.arange(n) % k
    return fold

def cv_accuracy(X, y, folds, predict_fn, rng=None, k=None):
    '''Balanced accuracy under k-fold CV. `folds` is a fold-assignment vector, or
    an int (new random folds are drawn). Reusing one fold vector across methods is
    what makes a comparison paired rather than noisy.'''
    folds = make_folds(len(y), folds, rng) if np.isscalar(folds) else np.asarray(folds)
    accs = []
    for f in np.unique(folds):
        te = folds == f
        yhat, yt = predict_fn(X[~te], y[~te], X[te]), y[te]
        accs.append(((yhat[yt == 0] == 0).mean() + (yhat[yt == 1] == 1).mean()) / 2)
    return float(np.mean(accs))

w_lda2, b_lda2 = lda_train(P_A, P_B, 0.0)
w_dom2 = P_B.mean(axis=0) - P_A.mean(axis=0)

acc_pair = cv_accuracy(np.vstack([P_A, P_B]),
                       np.concatenate([np.zeros(n_trials, int), np.ones(n_trials, int)]),
                       5, lambda a, b_, c: lda_predict(a, b_, c, 0.0), rng)
print(f"2-D example ({ori_C:g} vs {ori_D:g} deg) uses neurons "
      f"{pair[0]} ({pref_oris[pair[0]]:.1f}°) and {pair[1]} ({pref_oris[pair[1]]:.1f}°)")
print(f"This pair classifies at {100*acc_pair:.1f}% (5-fold cross-validated)")

In [ ]:
proj_A, proj_B = P_A @ w_lda2, P_B @ w_lda2
auc2 = auc_from_ranks(proj_A, proj_B)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6))

axes[0].plot(P_A[:,0], P_A[:,1], ".", ms=3, color=BLUE, label=f"{ori_C:g}°")
axes[0].plot(P_B[:,0], P_B[:,1], ".", ms=3, color=RED,  label=f"{ori_D:g}°")
mid = (P_A.mean(axis=0) + P_B.mean(axis=0)) / 2
for wv, col, name in [(w_dom2, "#f2a03d", "difference of means"), (w_lda2, "k", "LDA axis")]:
    v = wv / np.linalg.norm(wv) * 6
    axes[0].arrow(mid[0], mid[1], v[0], v[1], color=col, width=0.12,
                  length_includes_head=True, head_width=0.6, zorder=5, label=name)
xl = np.array([P_A[:,0].min(), P_A[:,0].max()])
axes[0].plot(xl, -(w_lda2[0]*xl + b_lda2)/w_lda2[1], "k-", lw=1.8)
axes[0].set(xlabel=f"Neuron {pair[0]} (pref {pref_oris[pair[0]]:.0f}°)",
            ylabel=f"Neuron {pair[1]} (pref {pref_oris[pair[1]]:.0f}°)", aspect="equal",
            title=f"Two neurons, two stimuli, one boundary ({100*acc_pair:.0f}%)")
axes[0].legend(fontsize=7, loc="upper right")

ed = np.linspace(min(proj_A.min(), proj_B.min()), max(proj_A.max(), proj_B.max()), 40)
axes[1].hist(proj_A, ed, color=BLUE, alpha=0.5)
axes[1].hist(proj_B, ed, color=RED,  alpha=0.5)
axes[1].set(xlabel="Projection onto LDA axis", ylabel="Trials",
            title=f"After projection: a 1-D problem, AUC = {auc2:.3f}")
fig.tight_layout()

The clouds are tilted along the diagonal because the shared gain fluctuation pushes both
neurons up and down together. The two arrows nearly coincide, and it is worth understanding
why: shared noise runs along $(+1,+1)$, while the signal is **opponent** — stimulus D drives
one neuron up and the other down, along $(-1,+1)$. Signal and noise are close to orthogonal,
so discounting the noise barely rotates the readout. Part VII constructs the opposite case.

Notice also what just happened. A two-dimensional classification problem became the
**one-dimensional signal detection problem of Part I**. That is true of every linear
classifier, and it is why Part I was worth the time: ROC, AUC and $d'$ apply unchanged to
the projected variable.

### Now the full population

In [ ]:
S_pool = pooled_cov(X_A, X_B)
ev = np.sort(np.linalg.eigvalsh(S_pool))[::-1]
print(f"Covariance matrix: {N} x {N}, condition number = {ev[0]/max(ev[-1], 1e-300):.3g}")

lam_lda = 0.05
w_lda, b_lda = lda_train(X_A, X_B, lam_lda)
w_dom = mu_B - mu_A

print(f"Training-set AUC, regularized LDA     : {auc_from_ranks(X_A@w_lda, X_B@w_lda):.4f}")
print(f"Training-set AUC, difference-of-means : {auc_from_ranks(X_A@w_dom, X_B@w_dom):.4f}")

If the number of trials per class is smaller than the number of neurons, $S$ is singular and
$S^{-1}$ does not exist. Even when it is invertible, the smallest eigenvalues are estimated
terribly, and inverting amplifies exactly those unreliable directions. The standard fix is
**shrinkage**:

$$S_{\text{reg}} = (1-\lambda)\,S + \lambda\,\overline{\mathrm{diag}(S)}\,I$$

which pulls the estimate toward a scaled identity. $\lambda = 0$ is plain LDA; $\lambda = 1$
ignores covariance entirely and reduces to the difference-of-means decoder.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0,0].semilogy(np.maximum(ev, 1e-12), "k", lw=1.5)
axes[0,0].set(xlabel="Eigenvalue index", ylabel="Eigenvalue",
              title="Spectrum of the within-class covariance")

axes[0,1].plot(pref_oris, w_lda/np.linalg.norm(w_lda), "k", lw=1.5, label="LDA")
axes[0,1].plot(pref_oris, w_dom/np.linalg.norm(w_dom), "--", lw=2, color=RED,
               label="difference of means")
axes[0,1].axhline(0, color=GREY, ls=":")
axes[0,1].set(xlabel="Preferred orientation (deg)", ylabel="Normalized weight", xlim=(0,180),
              title="Decoder weights as a function of preference")
axes[0,1].legend(fontsize=8)

sc_A, sc_B = X_A @ w_lda, X_B @ w_lda
ed = np.linspace(min(sc_A.min(), sc_B.min()), max(sc_A.max(), sc_B.max()), 40)
axes[1,0].hist(sc_A, ed, color=BLUE, alpha=0.5)
axes[1,0].hist(sc_B, ed, color=RED,  alpha=0.5)
axes[1,0].set(xlabel="LDA projection", ylabel="Trials",
              title=f"Population LDA, AUC = {auc_from_ranks(sc_A, sc_B):.3f}")

# how much regularization? split-half train/test
lam_grid = np.concatenate([[0], np.logspace(-4, 0, 25)])
trn, hold = np.arange(1, n_trials, 2), np.arange(0, n_trials, 2)
auc_train = np.zeros(lam_grid.size); auc_test = np.zeros(lam_grid.size)
for k, lam in enumerate(lam_grid):
    wk, _ = lda_train(X_A[trn], X_B[trn], lam)
    auc_train[k] = auc_from_ranks(X_A[trn] @ wk, X_B[trn] @ wk)
    auc_test[k]  = auc_from_ranks(X_A[hold] @ wk, X_B[hold] @ wk)

axes[1,1].semilogx(np.maximum(lam_grid, 1e-4), auc_train, "-o", ms=3, color=GREY, label="training")
axes[1,1].semilogx(np.maximum(lam_grid, 1e-4), auc_test,  "-o", ms=3, color=RED,  label="held out")
axes[1,1].set(xlabel="Shrinkage λ", ylabel="AUC", title="Regularization closes the train/test gap")
axes[1,1].legend(fontsize=8, loc="lower left")
fig.tight_layout()

The weight profile is the **derivative** of the tuning curve, with opposite signs on the two
flanks — the decoder implements exactly the opponent comparison the neurometric analysis
predicted: subtract the neurons that prefer A from those that prefer B.

The training curve decreases monotonically — less regularization always fits training data
better. The held-out curve is non-monotonic, peaking at intermediate $\lambda$. The vertical
gap between them is overfitting made visible.

### Checking against scikit-learn

scikit-learn's `LinearDiscriminantAnalysis` with `solver="lsqr"` and a `shrinkage` parameter
implements the same estimator. Its shrinkage convention matches ours, so the weight vectors
should be parallel — though not equal in length, since only the *direction* is determined.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

sk_lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=lam_lda).fit(X, y)
w_sk = sk_lda.coef_.ravel()

cos_sim = (w_lda @ w_sk) / (np.linalg.norm(w_lda) * np.linalg.norm(w_sk))
print(f"cosine similarity between our LDA weights and sklearn's: {cos_sim:.6f}")
print(f"angle between them: {np.degrees(np.arccos(np.clip(cos_sim, -1, 1))):.3f} deg")

> ### Homework question 4
> **(a)** Derive $\mathbf{w}\propto S^{-1}(\mu_B-\mu_A)$ by maximizing the Fisher criterion
> $(\mathbf{w}^\top(\mu_B-\mu_A))^2 / (\mathbf{w}^\top S\mathbf{w})$.
>
> **(b)** Show that when $S \propto I$, LDA and the difference-of-means decoder give the same
> boundary.
>
> **(c)** Reduce `n_trials` to 50 and re-run. What happens to the condition number, to the
> unregularized weights, and to the best $\lambda$?
>
> **(d)** LDA assumes both classes share one covariance. In our simulation variance scales
> with the mean, so they differ slightly. What would you use if the difference were large?
> *(Look up quadratic discriminant analysis.)*

---
## Part IV. Logistic regression: a GLM classifier

LDA is **generative**: it models the response distribution of each class and derives a
boundary. Logistic regression is **discriminative**: it models the class probability
directly and never describes the responses at all.

$$P(y = 1 \mid \mathbf{r}) = \frac{1}{1 + \exp[-(\mathbf{w}^\top\mathbf{r} + b)]}$$

Equivalently, the log-odds are linear in the population response. This is a generalized
linear model with a Bernoulli noise model and a logit link — the same GLM machinery used to
fit spike counts, pointed at a binary outcome.

We fit by maximizing the penalized log-likelihood using **iteratively reweighted least
squares** (Newton's method). The ridge penalty is not optional: with 80 correlated
predictors and separable training data, the unpenalized likelihood is maximized by weights
of infinite magnitude.

In [ ]:
def logistic_train(X, y, lam=1.0, max_iter=100, tol=1e-8):
    '''Ridge-penalized logistic regression by IRLS. Returns [intercept, *weights].
    The intercept is not penalized.'''
    n, p = X.shape
    Xa = np.column_stack([np.ones(n), X])
    w = np.zeros(p + 1)
    R = lam * np.eye(p + 1); R[0, 0] = 0.0
    for _ in range(max_iter):
        mu = 1 / (1 + np.exp(-(Xa @ w)))
        s = np.clip(mu * (1 - mu), 1e-6, None)
        g = Xa.T @ (y - mu) - R @ w                     # gradient
        H = Xa.T @ (Xa * s[:, None]) + R                # Hessian
        step = np.linalg.solve(H, g)
        w += step
        if np.linalg.norm(step) < tol:
            break
    return w

def logistic_predict_proba(X, w):
    return 1 / (1 + np.exp(-(np.column_stack([np.ones(len(X)), X]) @ w)))

lam_lr = 1.0
w_lr_full = logistic_train(X, y, lam_lr)
w_lr = w_lr_full[1:]
p_hat = logistic_predict_proba(X, w_lr_full)

print(f"Training-set AUC, ridge logistic: {auc_from_ranks(p_hat[y==0], p_hat[y==1]):.4f}")
cos_ang = (w_lr @ w_lda) / (np.linalg.norm(w_lr) * np.linalg.norm(w_lda))
print(f"Angle between logistic and LDA weight vectors: "
      f"{np.degrees(np.arccos(np.clip(cos_ang, -1, 1))):.1f} deg")

In [ ]:
def calibration_curve_manual(p, y, nbin=10):
    edges = np.linspace(0, 1, nbin + 1)
    ctr, obs = [], []
    for b in range(nbin):
        m = (p >= edges[b]) & (p < edges[b+1])
        if m.sum() > 0:
            ctr.append(p[m].mean()); obs.append(y[m].mean())
    return np.array(ctr), np.array(obs)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0,0].plot(pref_oris, w_lr/np.linalg.norm(w_lr), lw=1.8, color=PURPLE, label="logistic")
axes[0,0].plot(pref_oris, w_lda/np.linalg.norm(w_lda), "k--", lw=1.5, label="LDA")
axes[0,0].axhline(0, color=GREY, ls=":")
axes[0,0].set(xlabel="Preferred orientation (deg)", ylabel="Normalized weight", xlim=(0,180),
              title="Logistic and LDA weights are nearly proportional")
axes[0,0].legend(fontsize=8)

ed = np.linspace(0, 1, 30)
axes[0,1].hist(p_hat[y==0], ed, color=BLUE, alpha=0.5)
axes[0,1].hist(p_hat[y==1], ed, color=RED,  alpha=0.5)
axes[0,1].set(xlabel="Predicted P(stimulus = B)", ylabel="Trials",
              title="Logistic outputs are probabilities")

ctr, obs = calibration_curve_manual(p_hat, y)
axes[1,0].plot([0,1],[0,1], ":", color=GREY, lw=1.5)
axes[1,0].plot(ctr, obs, "-o", lw=2, color=PURPLE)
axes[1,0].set(xlabel="Predicted probability", ylabel="Observed fraction of class B",
              xlim=(0,1), ylim=(0,1), aspect="equal", title="Calibration")

lam_grid_lr = np.logspace(-2, 3, 20)
sub = np.concatenate([np.arange(1, n_trials, 2), n_trials + np.arange(1, n_trials, 2)])
w_path = np.array([logistic_train(X[sub], y[sub], lam)[1:] for lam in lam_grid_lr])
axes[1,1].semilogx(lam_grid_lr, w_path[:, np.round(np.linspace(0, N-1, 15)).astype(int)], lw=1.2)
axes[1,1].set(xlabel="Ridge penalty λ", ylabel="Weight",
              title="Regularization path (15 example neurons)")
fig.tight_layout()

The two weight profiles agree closely, and that is not a coincidence: if the classes really
are Gaussian with shared covariance, logistic regression and LDA estimate the same boundary.
They differ in what they *assume*, and therefore in how they fail. Logistic regression is
more robust to non-Gaussian responses and outliers; LDA is more efficient when its
assumptions hold, which matters when trials are scarce.

The probabilistic output is the practical advantage of the GLM. It lets you ask questions a
bare label cannot answer: on which trials was the population pattern ambiguous, and does the
animal's behavior on those trials track the ambiguity? Relating decoder confidence to choice
is a standard tool in perceptual decision-making work.

As $\lambda$ grows, all weights shrink smoothly toward zero. Ridge (L2) shrinks but never
zeroes them. An L1 (lasso) penalty would drive weights to exactly zero one by one, giving a
sparse decoder that reads out only a handful of neurons — often a more biologically
plausible hypothesis, and easier to interpret.

In [ ]:
# Check against scikit-learn. Note the parameterization: sklearn's C is the INVERSE
# of the regularization strength, so C = 1/lambda matches our formulation. Note also
# what happens at sklearn's DEFAULT convergence tolerance (1e-4): the weights agree
# well but the intercept does not, because a nearly balanced design leaves the
# intercept weakly determined and lbfgs stops early. Tighten the tolerance and the
# two fits agree to seven digits. Default tolerances are a real source of
# "why don't these two implementations match?".
from sklearn.linear_model import LogisticRegression

for tol in (1e-4, 1e-8):
    sk_lr = LogisticRegression(C=1/lam_lr, solver="lbfgs", max_iter=20000, tol=tol).fit(X, y)
    print(f"sklearn tol={tol:<6g} | max|Δweight| = {np.abs(w_lr - sk_lr.coef_.ravel()).max():.2e}"
          f" | intercept: ours {w_lr_full[0]:+.6f}, sklearn {sk_lr.intercept_[0]:+.6f}")

> ### Homework question 5
> **(a)** Write down the gradient and Hessian of the penalized log-likelihood and confirm
> they match the IRLS update in `logistic_train`.
>
> **(b)** Set `lam_lr = 0` and `n_trials = 30`, then re-fit. What happens to the magnitude of
> the weights, and why? *(This is called complete separation.)*
>
> **(c)** The model assumes log-odds are **linear** in firing rate. Name a situation in
> neural data where that is clearly wrong, and how you would modify the model.
>
> **(d)** Implement an L1 penalty (or use `penalty="l1", solver="liblinear"`) and compare the
> weight profile to the ridge solution.

---
## Part V. Support vector machines and the margin

LDA asks about distributions. Logistic regression asks about probabilities. The support
vector machine asks a purely **geometric** question: of all hyperplanes that separate the two
classes, which sits as far as possible from the nearest points of either class?

That distance is the **margin**, and the trials touching it are the **support vectors**.
Everything else could be deleted without changing the solution — very different from LDA,
where every trial contributes to the mean and covariance.

Real neural data is never perfectly separable, so we use the **soft-margin** formulation:

$$\min_{\mathbf{w},b}\ \tfrac12\|\mathbf{w}\|^2 + C\sum_t \max\big(0,\ 1 - y_t(\mathbf{w}^\top\mathbf{r}_t + b)\big)$$

The hinge loss penalizes points on the wrong side of the margin. Small $C$ means a wide
margin with many violations (heavily regularized); large $C$ a narrow margin that respects
the training data. $C$ is an *inverse* regularization strength, so it moves opposite to
$\lambda$.

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

P2 = np.vstack([P_A, P_B])
y2 = np.concatenate([np.zeros(len(P_A), int), np.ones(len(P_B), int)])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.6))

for ax, C_val in zip(axes, [0.01, 1.0]):
    mdl = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C_val)).fit(P2, y2)
    scaler, svc = mdl[0], mdl[1]
    # Undo standardization so weights live in raw response units
    wv = svc.coef_.ravel() / scaler.scale_
    bv = svc.intercept_[0] - np.sum(svc.coef_.ravel() * scaler.mean_ / scaler.scale_)
    sv = scaler.inverse_transform(svc.support_vectors_)

    ax.plot(P_A[:,0], P_A[:,1], ".", ms=3, color=BLUE)
    ax.plot(P_B[:,0], P_B[:,1], ".", ms=3, color=RED)
    ax.plot(sv[:,0], sv[:,1], "o", ms=6, mfc="none", mec="0.15", mew=0.7)
    xl = np.array([P2[:,0].min(), P2[:,0].max()])
    for off, style, col in [(0, "-", "k"), (-1, "-", "0.5"), (1, "-", "0.5")]:
        ax.plot(xl, -(wv[0]*xl + bv + off)/wv[1], style, color=col, lw=1.8 if off==0 else 1.2)
    ax.set(xlabel=f"Neuron {pair[0]}", ylabel=f"Neuron {pair[1]}", aspect="equal",
           ylim=(P2[:,1].min()-1, P2[:,1].max()+1),
           title=f"SVM, C = {C_val:g}  ({len(sv)} support vectors)")
fig.tight_layout()

Circled points are support vectors. With small $C$ the margin is wide, most of the data lies
inside it, so almost every trial is a support vector and the boundary reflects the bulk of
the data. With large $C$ the margin is narrow, few trials define the boundary, and the
solution is more sensitive to individual noisy trials.

### A linear kernel is not always enough

Consider discriminating **cardinal** orientations (0° and 90°) from **oblique** ones (45° and
135°), read out from two neurons preferring 22.5° and 67.5°. Those preferences sit exactly
between the stimuli, and the consequence is worth working out before you look at the plot:

| stimulus | neuron 22.5° | neuron 67.5° |
|---|---|---|
| 0° | high | low |
| 90° | low | high |
| 45° | high | high |
| 135° | low | low |

Each neuron alone is completely uninformative about the category — neuron 22.5° is high for
one cardinal (0°) and one oblique (45°). The category is the **exclusive OR** of the two
responses, and the four clusters sit at the corners of a square with the classes on opposite
diagonals. This is the canonical problem no straight line can solve, and it is not
contrived: it is what "the information is present but not linearly available" actually looks
like.

In [ ]:
gain_hi, n_tr_k = 20.0, 250        # higher contrast so the four clusters separate cleanly
pair_k = [int(np.argmin(np.abs(pref_oris - 22.5))), int(np.argmin(np.abs(pref_oris - 67.5)))]

Xc = np.vstack([sample_trials(tuning_mean(pref_oris, kappa, o, baseline, gain_hi),
                              n_tr_k, fano, rng) for o in (0, 90)])
Xo = np.vstack([sample_trials(tuning_mean(pref_oris, kappa, o, baseline, gain_hi),
                              n_tr_k, fano, rng) for o in (45, 135)])
K2 = np.vstack([Xc[:, pair_k], Xo[:, pair_k]])
yk = np.concatenate([np.zeros(len(Xc), int), np.ones(len(Xo), int)])

# Honest accuracies: fit on half the trials, score on the other half. The training
# accuracy of an RBF SVM is close to meaningless -- a narrow kernel can put a bubble
# around every training point and score 100% while learning nothing.
kf = make_folds(len(yk), 2, rng)
k_tr, k_te = kf == 0, kf == 1

lin = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)).fit(K2[k_tr], yk[k_tr])
rbf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")).fit(K2[k_tr], yk[k_tr])
acc_lin = lin.score(K2[k_te], yk[k_te])
acc_rbf = rbf.score(K2[k_te], yk[k_te])

fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot(K2[yk==0,0], K2[yk==0,1], ".", ms=3, color=BLUE, label="cardinal")
ax.plot(K2[yk==1,0], K2[yk==1,1], ".", ms=3, color=RED,  label="oblique")
gx = np.linspace(K2[:,0].min(), K2[:,0].max(), 200)
gy = np.linspace(K2[:,1].min(), K2[:,1].max(), 200)
GX, GY = np.meshgrid(gx, gy)
Z = rbf.decision_function(np.column_stack([GX.ravel(), GY.ravel()])).reshape(GX.shape)
ax.contour(GX, GY, Z, levels=[0], colors="k", linewidths=2)
ax.set(xlabel=f"Neuron {pair_k[0]} (pref {pref_oris[pair_k[0]]:.0f}°)",
       ylabel=f"Neuron {pair_k[1]} (pref {pref_oris[pair_k[1]]:.0f}°)", aspect="equal",
       title=f"Cardinal vs oblique (held out):\nlinear {acc_lin:.2f}, RBF {acc_rbf:.2f}")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()

The linear SVM sits at chance, as it must. The RBF kernel implicitly maps each trial into a
very high-dimensional feature space where the classes *do* become linearly separable, then
finds a maximum-margin hyperplane there. Projected back down, that hyperplane becomes the
closed contours around the cardinal clusters. The kernel trick means you never compute the
mapping — you only ever evaluate inner products
$k(\mathbf{r},\mathbf{r}') = \exp(-\gamma\|\mathbf{r}-\mathbf{r}'\|^2)$.

A word of caution before you reach for a kernel in real analyses. A nonlinear classifier
that beats a linear one tells you the information is present but not linearly available.
Whether a downstream neuron could actually extract it is a separate question, and most
readout hypotheses in systems neuroscience are deliberately linear for exactly that reason.

> ### Homework question 6
> **(a)** Explain why deleting all non-support-vector trials leaves the SVM unchanged, and
> why the same is not true for LDA.
>
> **(b)** Plot the hinge loss and the logistic loss against the margin $y f(\mathbf{r})$. How
> do they differ for confidently correct and for grossly misclassified points?
>
> **(c)** Vary `gamma` for the RBF SVM over orders of magnitude. What happens at very small
> and very large values? Which failure mode is overfitting?
>
> **(d)** Build a version of cardinal-vs-oblique that **is** linearly separable, by using
> neurons preferring 0° and 90° instead. Work out from the tuning curves why a straight line
> suddenly suffices. What does that say about the claim that a category is "nonlinearly
> encoded"?

---
## Part VI. Cross-validation, overfitting, and regularization

Every accuracy so far was measured on the same trials used to fit the classifier. **Those
numbers are not estimates of anything you care about.** A classifier with enough free
parameters can memorize noise, and with 80 neurons it has plenty.

The remedy is to evaluate on data the fit never saw. We implemented k-fold cross-validation
by hand above (`cv_accuracy`), because the procedure is short and worth seeing; scikit-learn's
`StratifiedKFold` and `cross_val_score` do the same thing in production code.

In [ ]:
kfold = 10
folds = make_folds(len(y), kfold, rng)     # draw ONCE, reuse for every method

acc_lda_cv = cv_accuracy(X, y, folds, lambda a,b_,c: lda_predict(a,b_,c,0.05))
acc_dom_cv = cv_accuracy(X, y, folds, lambda a,b_,c: lda_predict(a,b_,c,1.0))
acc_lr_cv  = cv_accuracy(X, y, folds,
                         lambda a,b_,c: logistic_predict_proba(c, logistic_train(a,b_,1.0)) > 0.5)

print(f"{kfold}-fold cross-validated accuracy:")
print(f"  regularized LDA     : {acc_lda_cv:.3f}")
print(f"  difference of means : {acc_dom_cv:.3f}")
print(f"  ridge logistic      : {acc_lr_cv:.3f}")

In [ ]:
# --- Overfitting made explicit: vary the number of training trials ---
# A single draw of the training set is very noisy, so we average over several
# random draws. The last 100 trials of each class are always held out.
test_idx = np.arange(n_trials - 100, n_trials)
n_train_grid = np.array([5, 10, 20, 30, 40, 60, 80, 120, 200, 300])
n_reps = 20

acc_tr = np.zeros((n_reps, n_train_grid.size))
acc_te = np.zeros((n_reps, n_train_grid.size))
acc_te_reg = np.zeros((n_reps, n_train_grid.size))
pool = np.arange(n_trials - 100)                      # trials eligible for training

for r in range(n_reps):
    order = rng.permutation(pool)
    for k, nt in enumerate(n_train_grid):
        sel = order[:nt]
        trA, trB = X_A[sel], X_B[sel]
        teA, teB = X_A[test_idx], X_B[test_idx]
        for lam, store_tr in [(0.0, True), (0.2, False)]:
            wk, bk = lda_train(trA, trB, lam)
            te = ((teA @ wk + bk < 0).mean() + (teB @ wk + bk > 0).mean()) / 2
            if store_tr:
                acc_te[r, k] = te
                acc_tr[r, k] = ((trA @ wk + bk < 0).mean() + (trB @ wk + bk > 0).mean()) / 2
            else:
                acc_te_reg[r, k] = te

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

axes[0].semilogx(n_train_grid, acc_tr.mean(0), "-o", lw=2, color=GREY, label="training")
axes[0].semilogx(n_train_grid, acc_te.mean(0), "-o", lw=2, color=RED,  label="test")
axes[0].fill_between(n_train_grid, acc_te.mean(0)-acc_te.std(0),
                     acc_te.mean(0)+acc_te.std(0), color=RED, alpha=0.15)
axes[0].axhline(0.5, color=GREY, ls=":")
axes[0].axvline(N, color="k", ls="--", lw=1)
axes[0].text(N*1.15, 0.36, f"N = {N}\nneurons", fontsize=7)
axes[0].set(xlabel="Training trials per class", ylabel="Accuracy", ylim=(0.3, 1.05),
            title=f"Learning curve, unregularized LDA (mean of {n_reps} draws)")
axes[0].legend(fontsize=8, loc="center right")

axes[1].semilogx(n_train_grid, acc_te.mean(0),     "-o", lw=2, color=RED,   label="λ = 0")
axes[1].semilogx(n_train_grid, acc_te_reg.mean(0), "-s", lw=2, color=GREEN, label="λ = 0.2")
axes[1].axhline(0.5, color=GREY, ls=":")
axes[1].set(xlabel="Training trials per class", ylabel="Test accuracy", ylim=(0.3, 1.05),
            title="Regularization rescues the small-sample regime")
axes[1].legend(fontsize=8, loc="lower right")
fig.tight_layout()

The left panel is the single most important figure in the tutorial. When training trials are
fewer than neurons, unregularized LDA classifies the training data almost perfectly while
performing barely above chance on held-out trials. It has found a direction that separates
*these particular noise samples*, and that direction means nothing.

Look at where test accuracy is **worst**. The dip sits at 40 trials per class — that is 80
training trials in total, exactly the number of neurons. This is the *interpolation
threshold*: the point at which the model has just enough freedom to fit the training data
exactly and no more. There the covariance estimate is maximally ill-conditioned, $S^{-1}$
blows up along its smallest eigendirections, and generalization is at its worst. Performance
recovers on both sides of it — with fewer trials the pseudoinverse's minimum-norm solution is
implicitly regularized, and with more trials the estimate becomes genuinely good. Machine
learning calls this shape *double descent*; here it falls out of nothing more exotic than
inverting a badly estimated covariance matrix.

Notice also that the two curves converge from opposite directions: training accuracy falls as
test accuracy rises.

If you have ever seen a decoding result reported without cross-validation, this figure is why
you should not believe it.

### Choosing $\lambda$ honestly

The temptation is to pick the $\lambda$ that maximizes cross-validated accuracy and then
report that accuracy. That is a subtle form of overfitting: you used the test folds to make a
modeling decision. The correct procedure **nests** an inner cross-validation loop inside the
outer one.

In [ ]:
def nested_cv_accuracy(X, y, kfold, lam_grid, rng):
    '''Inner loop selects lambda; outer loop measures accuracy. The honest way to
    report a tuned classifier.'''
    folds = make_folds(len(y), kfold, rng)
    accs = []
    for f in np.unique(folds):
        te = folds == f
        Xtr, ytr = X[~te], y[~te]
        inner = [cv_accuracy(Xtr, ytr, 4, lambda a,b_,c,l=lam: lda_predict(a,b_,c,l), rng)
                 for lam in lam_grid]
        best = lam_grid[int(np.argmax(inner))]
        yhat, yt = lda_predict(Xtr, ytr, X[te], best), y[te]
        accs.append(((yhat[yt==0]==0).mean() + (yhat[yt==1]==1).mean()) / 2)
    return float(np.mean(accs))

lam_cv = np.array([0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0])

# The optimism bias is small, and a SINGLE comparison is swamped by fold-to-fold
# noise -- so we repeat both procedures and compare their averages. Measuring a
# small effect once and reporting the sign is exactly the mistake this section
# is about.
n_rep_cv = 8
single_loop = np.zeros(n_rep_cv)
nested = np.zeros(n_rep_cv)
for r in range(n_rep_cv):
    fo = make_folds(len(y), 5, rng)
    single_loop[r] = max(cv_accuracy(X, y, fo, lambda a,b_,c,l=lam: lda_predict(a,b_,c,l))
                         for lam in lam_cv)
    nested[r] = nested_cv_accuracy(X, y, 5, lam_cv, rng)

print(f"single-loop CV, best lambda : {single_loop.mean():.4f} ± {single_loop.std():.4f}")
print(f"nested CV                   : {nested.mean():.4f} ± {nested.std():.4f}")
print(f"optimism bias               : {single_loop.mean() - nested.mean():+.4f}")
print(f"single > nested in {(single_loop > nested).sum()} of {n_rep_cv} repeats")

acc_by_lam = np.array([cv_accuracy(X, y, folds, lambda a,b_,c,l=lam: lda_predict(a,b_,c,l))
                       for lam in lam_cv])

# --- A null distribution: how good is 'good'? ---
n_perm = 50
acc_null = np.array([cv_accuracy(X, rng.permutation(y), 5,
                                 lambda a,b_,c: lda_predict(a,b_,c,0.05), rng)
                     for _ in range(n_perm)])
p_value = (1 + (acc_null >= acc_lda_cv).sum()) / (1 + n_perm)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(lam_cv, acc_by_lam, "-o", lw=2, color=PURPLE)
axes[0].set(xlabel="Shrinkage λ", ylabel="CV accuracy",
            title="Cross-validated accuracy vs. regularization")
axes[1].hist(acc_null, 15, color="0.6")
axes[1].axvline(acc_lda_cv, color="r", lw=2)
axes[1].set(xlabel="CV accuracy", ylabel="Count",
            title=f"Label-shuffled null,  p < {p_value:.3f}")
fig.tight_layout()

The optimism bias is **real but small here** — well under one percentage point — and it is
smaller than the spread across repeats. That is why a single run of each procedure can easily
show the *wrong* ordering, and why the code above averages several repeats before comparing.

Do not read that as "nesting does not matter". The bias grows with the number of
hyperparameters you tune, the size of the grid, and the smallness of the dataset. Here we
tune one parameter over eight values with 800 trials, which is close to the best case. Tune
five hyperparameters on 60 trials and the optimism can be enormous. The general rule stands:
anything chosen by looking at held-out data must itself be validated on data you have not
looked at.

There is a second lesson buried in those error bars. A cross-validated accuracy quoted to
three decimals from a single run is mostly noise in the last two. When you report a decoding
accuracy, report its spread too.

The null distribution is centered on 0.5 but has real **width**. With few trials that width
can be large, and a decoder that looks impressive may fall well inside it. An accuracy of
0.62 can be highly significant or completely unremarkable depending on how many trials you
have — the permutation test is what tells you which.

> ### Homework question 7
> **(a)** The learning curve above averages 20 random draws of the training set. Re-run it
> with `n_reps = 1` a few times. How much does the curve move? What does that tell you about
> published learning curves shown without error bars?
>
> **(a2)** Predict where the interpolation dip would move if you halved the number of
> neurons to 40, then check.
>
> **(b)** Explain in your own words why the single-loop CV number is optimistic. How large is
> the bias here, and when would it be worse?
>
> **(c)** Our folds shuffle trials randomly. In a real experiment with slow drift in the
> recording, why is that too permissive, and what would you do instead?
>
> **(d)** Increase `n_perm` to 500 for an exact p-value. How does the width of the null
> depend on the number of trials?

---
## Part VII. Decoding geometry: signal vs. noise correlations

Everything so far assumed independent noise. Real cortical neurons are correlated: pairs
with similar preferred orientations share more noise than pairs with different preferences.
Whether correlations help or hurt depends entirely on their **geometry relative to the
signal**.

We generate limited-range correlations, $c_{ij} = c_0 \exp(-|\phi_i - \phi_j|/\tau)$, and
compare decoding with and without them.

*Note on $c_0$:* measured spike-count correlations in visual cortex are usually 0.1–0.2 for
nearby, similarly tuned pairs. We use a larger value so the effect is visible in a single
simulation rather than only in an average over many. Homework question 8 asks you to repeat
this with realistic values.

In [ ]:
c0, tau = 0.5, 25.0

def limited_range_corr(pref_oris, c0, tau_deg):
    d = np.abs(pref_oris[:, None] - pref_oris[None, :])
    d = np.minimum(d, 180 - d)                      # circular distance on 180 deg
    C = c0 * np.exp(-d / tau_deg)
    np.fill_diagonal(C, 1.0)
    return (C + C.T) / 2

def nearest_pd(C, floor=1e-6):
    vals, vecs = np.linalg.eigh((C + C.T)/2)
    A = vecs @ np.diag(np.maximum(vals, floor)) @ vecs.T
    return (A + A.T) / 2

C_lr = limited_range_corr(pref_oris, c0, tau)
L_lr = np.linalg.cholesky(nearest_pd(C_lr))         # LOWER factor; see sample_trials

XA_c = sample_trials(mu_A, n_trials, fano, rng, chol_lower=L_lr)
XB_c = sample_trials(mu_B, n_trials, fano, rng, chol_lower=L_lr)

def dprime_along(X0, X1, w):
    p0, p1 = X0 @ w, X1 @ w
    return abs(p1.mean() - p0.mean()) / np.sqrt((p0.var(ddof=1) + p1.var(ddof=1))/2)

w_dom_n = (mu_B - mu_A) / np.linalg.norm(mu_B - mu_A)
d_ind_dom = dprime_along(X_A,  X_B,  w_dom_n)
d_cor_dom = dprime_along(XA_c, XB_c, w_dom_n)
w_i, _ = lda_train(X_A,  X_B,  0.05)
w_c, _ = lda_train(XA_c, XB_c, 0.05)
d_ind_opt = dprime_along(X_A,  X_B,  w_i/np.linalg.norm(w_i))
d_cor_opt = dprime_along(XA_c, XB_c, w_c/np.linalg.norm(w_c))

print(f"Optimal / naive d-prime, independent noise : {d_ind_opt/d_ind_dom:.3f}")
print(f"Optimal / naive d-prime, correlated noise  : {d_cor_opt/d_cor_dom:.3f}")

Two things happen when correlations are introduced. Both decoders lose $d'$: shared noise is
not averaged away by pooling, so the population is simply less informative. And — more
interesting — the **advantage** of the optimal decoder over the naive one grows. With
independent noise, knowing the covariance buys a few percent; with strong correlations the
$S^{-1}$ earns its keep by steering the readout away from noisy directions.

That is a matter of degree. The next simulation shows a case where the loss is **absolute**.

### Information-limiting correlations

The geometry that truly matters is whether noise lies **along the signal direction**. Noise
in any other direction can be projected out by a suitable decoder. Noise parallel to
$\mu_B - \mu_A$ cannot be, no matter how many neurons you record, because it is
indistinguishable from a change in the stimulus.

We add a rank-one noise component along the signal axis and sweep its amplitude, using the
analytic optimum $d'^2 = \Delta\mu^\top S^{-1} \Delta\mu$.

In [ ]:
df = mu_B - mu_A
f_prime = df / np.linalg.norm(df)
eps_grid = np.concatenate([[0], np.logspace(-3, 0, 12)])
S_base = np.diag(fano * (mu_A + mu_B) / 2)

d_opt_grid = np.zeros(eps_grid.size)   # N neurons
d_opt_n    = np.zeros(eps_grid.size)   # 4N neurons (noise reduced fourfold)
for k, eps in enumerate(eps_grid):
    rank1 = eps * np.linalg.norm(df)**2 * np.outer(f_prime, f_prime)
    d_opt_grid[k] = np.sqrt(df @ np.linalg.solve(S_base + rank1, df))
    d_opt_n[k]    = np.sqrt(df @ np.linalg.solve(S_base/4 + rank1, df))

# Centre each class separately to see the NOISE covariance, not the stimulus difference.
# Computing PCs by SVD keeps the link to the linear algebra tutorial explicit: the right
# singular vectors of the centred data matrix are the eigenvectors of its covariance.
Xc_cent = np.vstack([XA_c - XA_c.mean(axis=0), XB_c - XB_c.mean(axis=0)])
_, Sv, Vt = np.linalg.svd(Xc_cent, full_matrices=False)
coeff_noise = Vt.T
lat_noise = Sv**2 / (len(Xc_cent) - 1)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

im = axes[0,0].imshow(C_lr, extent=[0,180,180,0], cmap="viridis")
axes[0,0].set(xlabel="Preferred ori (deg)", ylabel="Preferred ori (deg)",
              title="Noise correlation matrix"); axes[0,0].grid(False)
fig.colorbar(im, ax=axes[0,0], fraction=0.046)

xb = np.arange(2)
axes[0,1].bar(xb-0.2, [d_ind_dom, d_cor_dom], 0.4, color=BLUE, label="difference-of-means")
axes[0,1].bar(xb+0.2, [d_ind_opt, d_cor_opt], 0.4, color=RED,  label="LDA (optimal)")
axes[0,1].set_xticks(xb); axes[0,1].set_xticklabels(["independent", "correlated"])
axes[0,1].set(ylabel="d'", title="Correlations penalize the naive decoder")
axes[0,1].legend(fontsize=8)

axes[0,2].loglog(np.maximum(eps_grid,1e-4), d_opt_grid, "-o", lw=2, color=BLUE, label="N neurons")
axes[0,2].loglog(np.maximum(eps_grid,1e-4), d_opt_n,    "-s", lw=2, color=RED,  label="4N neurons")
axes[0,2].set(xlabel="Amplitude of signal-aligned noise ε", ylabel="Optimal d'",
              title="Information-limiting correlations saturate")
axes[0,2].legend(fontsize=8, loc="lower left")

n_show = 20
overlap = np.abs(coeff_noise[:, :n_show].T @ f_prime)
axes[1,0].bar(np.arange(1, n_show+1), overlap, color=PURPLE)
axes[1,0].set(xlabel="Noise PC index", ylabel="|cos angle with signal axis|",
              title="The signal is not aligned with the top noise PCs")

axes[1,1].plot(np.arange(1, n_show+1), 100*lat_noise[:n_show]/lat_noise.sum(), "-o", color="k")
axes[1,1].set(xlabel="Noise PC index", ylabel="Variance explained (%)",
              title="Noise variance spectrum")

pc1 = coeff_noise[:, 0]
if pc1.sum() < 0: pc1 = -pc1            # sign of an eigenvector is arbitrary
axes[1,2].plot(pref_oris, pc1, lw=1.8, color=PURPLE, label="noise PC1")
axes[1,2].plot(pref_oris, f_prime, "k--", lw=2, label="signal axis")
axes[1,2].axhline(0, color=GREY, ls=":")
axes[1,2].set(xlabel="Preferred orientation (deg)", ylabel="Normalized loading", xlim=(0,180),
              title="PC1 of noise vs. the discriminative direction")
axes[1,2].legend(fontsize=8)
fig.tight_layout()

With no signal-aligned noise, quadrupling the population doubles $d'$, exactly as independent
pooling predicts. With signal-aligned noise the two curves converge: adding neurons stops
helping. The population has an **information ceiling** no amount of extra recording can
raise. This is the central result of Moreno-Bote et al. (2014), and it is why "how many
neurons do I need?" is not a well-posed question without knowing the noise geometry.

The bottom row is the punchline that separates PCA from classification. The largest source of
population variance is a broad, mostly positive mode — a shared gain fluctuation that raises
or lowers the whole population together. The discriminative direction is the opponent,
sign-flipping profile from Part III. They are close to orthogonal.

**So if you had run PCA and kept the top few components, you would have thrown away most of
the stimulus information**, because the signal here is small in variance but large in
discriminability. Dimensionality reduction chosen by variance is not dimensionality reduction
chosen by relevance.

> ### Homework question 8
> **(a)** Derive $d'^2 = \Delta\mu^\top S^{-1}\Delta\mu$ for the optimal linear decoder of
> two Gaussians with common covariance $S$.
>
> **(b)** Explain geometrically why noise along $f'$ cannot be removed by any linear readout,
> while noise orthogonal to $f'$ can.
>
> **(c)** Set `c0` to a realistic cortical value (0.1–0.15) and re-run. Is the optimal
> decoder's advantage still detectable in one simulation? How many repeats would you need?
>
> **(d)** Set `c0` negative. Does discriminability go up or down? Why do some authors argue
> correlations are harmful while others argue they can be beneficial?
>
> **(e)** Repeat the noise-PC analysis with independent noise. Where does the signal axis sit
> in the PC ordering then?

---
## Part VIII. Nonlinear classifiers and a head-to-head comparison

We close by putting every method on the same footing: identical trials, identical folds,
accuracy only on held-out data.

We add one nonlinear method, **k-nearest neighbours**, which has no training step at all: to
classify a new trial, find the $k$ most similar training trials in the $N$-dimensional
response space and take a majority vote. It makes no assumption about the shape of the
boundary — both its strength and its weakness.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def knn_predict(Xtr, ytr, Xte, k=15):
    return KNeighborsClassifier(n_neighbors=k).fit(Xtr, ytr).predict(Xte)

def svm_predict(Xtr, ytr, Xte, C=1.0):
    return make_pipeline(StandardScaler(), SVC(kernel="linear", C=C)).fit(Xtr, ytr).predict(Xte)

methods = {
    "diff-of-means":    lambda a,b_,c: lda_predict(a,b_,c,1.0),
    "LDA (λ=0.05)":     lambda a,b_,c: lda_predict(a,b_,c,0.05),
    "ridge logistic":   lambda a,b_,c: logistic_predict_proba(c, logistic_train(a,b_,1.0)) > 0.5,
    "kNN (k=15)":       knn_predict,
    "linear SVM":       svm_predict,
}
acc_cmp = {name: cv_accuracy(X, y, folds, fn) for name, fn in methods.items()}
for name, a in acc_cmp.items():
    print(f"{name:<18}: {a:.3f}")

In [ ]:
# --- Where does kNN break down? ---
# In a real recording most neurons are not tuned to the variable you are decoding.
# We simulate that by appending UNINFORMATIVE neurons: same mean under both stimuli,
# same noise, carrying no signal. A good decoder should ignore them.
mu_flat = np.mean((mu_A + mu_B) / 2)
n_dist_grid = [0, 20, 50, 100, 200, 400]
lam_show = [0.05, 0.5, 0.95]
acc_lin_D = np.zeros((len(lam_show), len(n_dist_grid)))
acc_knn_D = np.zeros(len(n_dist_grid))
folds5 = make_folds(len(y), 5, rng)

for k, M in enumerate(n_dist_grid):
    Xd = X if M == 0 else np.hstack([X, sample_trials(np.full(M, mu_flat), len(y), fano, rng)])
    for j, lam in enumerate(lam_show):
        acc_lin_D[j, k] = cv_accuracy(Xd, y, folds5, lambda a,b_,c,l=lam: lda_predict(a,b_,c,l))
    acc_knn_D[k] = cv_accuracy(Xd, y, folds5, knn_predict)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

names = list(acc_cmp)
axes[0].bar(range(len(names)), [acc_cmp[n] for n in names], color="#5a72b5")
axes[0].axhline(0.5, color="k", ls=":", lw=1.5)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=30, ha="right")
axes[0].set(ylabel=f"{kfold}-fold CV accuracy", ylim=(0.4, 1),
            title="All methods, identical folds")

for j, (lam, col) in enumerate(zip(lam_show, [BLUE, GREEN, PURPLE])):
    axes[1].plot(n_dist_grid, acc_lin_D[j], "-o", lw=2, color=col, label=f"LDA λ={lam}")
axes[1].plot(n_dist_grid, acc_knn_D, "-s", lw=2, color=RED, label="kNN (k=15)")
axes[1].axhline(0.5, color="k", ls=":")
axes[1].set(xlabel="Number of uninformative neurons added", ylabel="CV accuracy",
            ylim=(0.5, 0.9), title="Curse of dimensionality")
axes[1].legend(fontsize=8, loc="lower left")
fig.tight_layout()

print(f"After adding {n_dist_grid[-1]} uninformative neurons:")
for j, lam in enumerate(lam_show):
    print(f"  LDA λ = {lam:<4} : {acc_lin_D[j,0]:.3f} -> {acc_lin_D[j,-1]:.3f}")
print(f"  kNN k = 15   : {acc_knn_D[0]:.3f} -> {acc_knn_D[-1]:.3f}")

Three things are visible at once, and together they are the practical summary of this
tutorial.

**Lightly regularized LDA collapses.** With hundreds of features and only 640 training trials
per fold, $\lambda = 0.05$ is not enough regularization, and the decoder starts fitting the
useless dimensions. It loses roughly ten points.

**Heavily regularized LDA degrades gracefully.** The same method at $\lambda = 0.5$ or $0.95$
loses only a few, because shrinkage lets it assign the useless neurons small weights. Notice
too how the *gap* between the $\lambda$ values widens from left to right: with 80 informative
neurons the choice of $\lambda$ barely matters, while with 480 mostly-useless ones it is the
difference between a usable decoder and a bad one. The best $\lambda$ is not a property of
the method — it depends on the dimensionality, so a value tuned on one dataset must be
re-tuned when that changes.

The same point explains a result that may look odd in the left panel: the plain
difference-of-means decoder does as well as anything else. It is simply LDA at
$\lambda = 1$ — maximal shrinkage — and on this problem, where the true optimal readout is
close to the difference of means and covariance is hard to estimate from 720 training trials,
throwing the covariance away entirely costs nothing. Sophistication is not automatically an
advantage.

**kNN degrades no matter what.** It has no weights to shrink: every neuron contributes
equally to the Euclidean distance, so uninformative dimensions inject noise directly into the
notion of "nearest". Push far enough and every training trial is roughly equidistant from
every test trial.

This is why methods that make an explicit assumption about the form of the boundary tend to
beat assumption-free methods on neural data, where the number of recorded units is large and
the number of trials is not.

Two further methods you will meet but which we do not implement here: **random forests**
(`sklearn.ensemble.RandomForestClassifier`) — ensembles of decision trees, robust, handle
nonlinearity automatically, and give a natural measure of which neurons matter; and **neural
networks**, overkill for a two-class problem with a few hundred trials, but the natural choice
when the decoded variable is continuous, high-dimensional, or unfolds over time.

A closing methodological point. In this simulation the optimal decoder is linear *by
construction*, so the elaborate methods cannot beat regularized LDA. That is often true of
real data too. When a fancy classifier outperforms a simple one on neural data, the first
question is not "what did the network learn?" but "did I cross-validate correctly, and is the
extra accuracy larger than the spread across folds?"

> ### Homework question 9
> **(a)** Add error bars to the comparison by reporting the standard deviation of accuracy
> **across folds**. Are any differences between methods larger than the fold-to-fold spread?
>
> **(b)** Sweep $k$ in kNN from 1 to 200. Sketch the bias–variance trade-off the curve
> reveals. Does the best $k$ change when the uninformative neurons are added?
>
> **(c)** Add an RBF-kernel SVM to the comparison. Does it beat the linear methods here?
> Should it, given how the data were generated?
>
> **(d)** Repeat the comparison using the correlated responses `XA_c` / `XB_c` from Part VII.
> Which methods suffer most, and does that match your prediction from the geometry?

> ### Homework question 10 — synthesis
> You record 150 neurons in V1 while a mouse discriminates 88° from 92°, and you obtain 60
> trials per condition. Design the complete analysis: which classifier, what regularization,
> how you would choose it, how you would cross-validate, what null distribution you would
> use, and what control analysis would convince a skeptical reviewer that your decoder reads
> out orientation rather than running speed, pupil size, or slow drift in the recording.

---
## Summary

1. **Signal detection theory** turns a one-neuron discrimination into a criterion-free
   summary: the ROC curve and its area. AUC equals the probability of a correct 2AFC
   judgment; $d'$ is a parametric shortcut valid only for equal-variance Gaussians.

2. **Information lives in the tuning-curve slope, not the peak.** The neurons that respond
   most strongly to both stimuli are the least informative about which one occurred.

3. **Every linear classifier reduces to Part I** after projection. LDA, logistic regression
   and the linear SVM differ only in how they choose the projection — by discounting noise
   covariance, by modelling class probability, or by maximizing a geometric margin — and on
   well-behaved data they land in nearly the same place.

4. **Cross-validation is not optional.** With more neurons than trials, an unregularized
   decoder achieves perfect training accuracy and chance test accuracy. Regularization
   rescues the small-sample regime, and the amount needed grows with dimensionality.

5. **Noise geometry sets the ceiling.** Correlations aligned with the signal direction cannot
   be removed by any readout, so the population saturates no matter how many neurons you add.
   And the discriminative direction is typically near-orthogonal to the largest principal
   components — variance is not relevance.

### Further reading

- Britten, Shadlen, Newsome & Movshon (1992). The analysis of visual motion: a comparison of
  neuronal and psychophysical performance. *Journal of Neuroscience* **12**, 4745–4765.
- Green & Swets (1966). *Signal Detection Theory and Psychophysics.*
- Averbeck, Latham & Pouget (2006). Neural correlations, population coding and computation.
  *Nature Reviews Neuroscience* **7**, 358–366.
- Moreno-Bote et al. (2014). Information-limiting correlations. *Nature Neuroscience* **17**,
  1410–1417.
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, chapters 4
  and 7.